# HockeyIQ Kenya — FIH World Cup 2026 Live Update

A separate, fast, standalone notebook \u2014 deliberately kept apart from
the main pipeline. The FIH Hockey World Cup 2026 (Belgium/Netherlands,
15\u201330 August) runs for just over two weeks, and the main weekly
automation would only refresh this data once, maybe twice, across the
whole tournament \u2014 not useful for something fans would want to
actually follow. This notebook scrapes only the World Cup standings
(fast, one page) and skips the much slower full domestic re-scrape
entirely, so it can run several times a day without wasting time or
GitHub Actions minutes on data that hasn't changed.


## Setup

In [1]:
# Cell 0 - Install Dependencies Not in the Standard Anaconda/Jupyter Setup

# pdfplumber isn\'t part of a typical Python or Anaconda install, unlike
# pandas/selenium/etc — confirmed by a real run hitting
# "ModuleNotFoundError: No module named \'pdfplumber\'" on a fresh
# environment. Running this cell first makes the notebook self-contained
# rather than relying on a separate manual pip install step, which is
# easy to miss.
#
# Uses subprocess directly (plain Python) rather than a Jupyter "!pip
# install" magic command — the two behave the same inside a notebook,
# but plain Python is real, testable code, not notebook-only syntax.
# Safe to re-run: pip simply reports "already satisfied" if a package
# is already installed.

import subprocess
import sys

def ensure_installed(package):
    try:
        __import__(package)
        print(f"{package} already installed.")
    except ImportError:
        print(f"Installing {package}...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package])
        except subprocess.CalledProcessError:
            print(f"\n\u26a0 Automatic install failed for {package}. This can happen in some "
                  f"restricted Python environments. Try running this in a terminal instead:\n"
                  f"    pip install {package}\n"
                  f"or, if you see an \'externally managed environment\' error specifically:\n"
                  f"    pip install --break-system-packages {package}")
            raise

ensure_installed("pdfplumber")
ensure_installed("requests")

print("\nDependencies ready.")


pdfplumber already installed.
requests already installed.

Dependencies ready.


In [2]:
# Cell 1 - Imports

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
import time
import io

print("Imports ready.")


Imports ready.


In [3]:
# Cell 2 - Configure Chrome

HEADLESS = True

chrome_options = Options()

if HEADLESS:
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--window-size=1920,1080")

chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option("useAutomationExtension", False)


def create_driver():
    d = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=chrome_options
    )
    d.execute_script("""
    Object.defineProperty(navigator, 'webdriver', {
        get: () => undefined
    })
    """)
    return d


driver = create_driver()

print(f"Chrome launched successfully. (headless={HEADLESS})")


Chrome launched successfully. (headless=True)


## Module: FIH World Cup 2026 Standings \u2014 "Closing the Gap" Context

Confirmed directly by inspecting the real event page: the current FIH
Hockey World Cup (Belgium/Netherlands 2026) has separate Men's and
Women's Points Tables, each split into four pools (A\u2013D). Kenya is not
among the competing nations \u2014 confirmed directly, South Africa is the
nearest African qualifier \u2014 so this data exists purely to give
Kenyan fans real, current context for where the global game stands,
not to track Kenya's own results.

**Honest note: as of this scrape, the tournament has only just begun
and most pool tables may still show 0\u20130 across the board.** This is
expected, not a scraping failure \u2014 the same weekly automation that
re-scrapes Kenya's own data will pick up real standings as the
tournament progresses.


In [4]:
# Cell WC1 - World Cup Standings Scraper

# Real elements found on a live run using a plain text search, that
# are never the actual clickable control: raw HTML/JS document
# metadata, and text labels on the match-schedule widget that just
# describe which pool a fixture belongs to, not a navigation control.
NON_INTERACTIVE_TAGS = {"script", "style", "title", "meta", "link", "noscript"}
NON_INTERACTIVE_CLASSES = {"team-time-text", "pool", "venue"}


def find_real_tab(text):
    """Confirmed real bug, fixed: searching for any element containing
    given text found the browser tab\'s <title> metadata before ever
    reaching the real gender toggle, and found <p class="pool venue">
    schedule labels instead of the real pool tabs \u2014 both are
    genuine elements that contain the right text, but neither is a
    clickable control. This filters to elements that are visible on
    screen, are not known non-interactive tags, and prefers a real
    button/link/tab-role element when one exists among the matches,
    rather than trusting the first text match found."""
    candidates = driver.find_elements(By.XPATH, f"//*[contains(text(), \"{text}\")]")
    visible_real = []
    for el in candidates:
        try:
            tag = el.tag_name.lower()
            classes = (el.get_attribute("class") or "").lower()
            if tag in NON_INTERACTIVE_TAGS:
                continue
            if any(c in classes for c in NON_INTERACTIVE_CLASSES):
                continue
            if not el.is_displayed():
                continue
            visible_real.append(el)
        except Exception:
            continue

    if not visible_real:
        return None

    # Among the genuinely visible, interactive candidates, prefer an
    # actual button/link/tab-role element if one exists.
    for el in visible_real:
        tag = el.tag_name.lower()
        role = (el.get_attribute("role") or "").lower()
        if tag in ("button", "a") or role == "tab":
            return el
    return visible_real[0]


def robust_click_text(text):
    """Shared helper used by every click point in this notebook. Finds
    the real, visible, interactive element for the given text (see
    find_real_tab above), then scrolls it into view and falls back to
    a JavaScript click if a normal one doesn\'t register."""
    el = find_real_tab(text)
    if el is None:
        raise ValueError(f"No real clickable element found for text: {text!r}")
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", el)
    time.sleep(0.3)
    try:
        el.click()
    except Exception:
        driver.execute_script("arguments[0].click();", el)
    return el


WORLD_CUP_URL = "https://www.fih.hockey/events/fih-hockey-worldcup-belgium-netherlands-2026"

def scrape_world_cup_pool(pool_letter):
    """Extracts whichever pool table is currently visible on the page \u2014
    caller is responsible for clicking to the right gender/pool tab first."""
    page_text = BeautifulSoup(driver.page_source, "html.parser").get_text(" ", strip=True)
    # Confirmed real row format: "{rank} {country} {P} {W} {D} {L} {ScF} {ScA} {PDiff} {Pt}"
    row_pattern = re.compile(
        r"(\d)\s+([A-Za-z][A-Za-z .]+?)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(-?\d+)\s+(\d+)"
    )
    rows = []
    for m in row_pattern.finditer(page_text):
        rows.append({
            "PoolRank": int(m.group(1)), "Team": m.group(2).strip(), "Played": int(m.group(3)),
            "Won": int(m.group(4)), "Drawn": int(m.group(5)), "Lost": int(m.group(6)),
            "ScoredFor": int(m.group(7)), "ScoredAgainst": int(m.group(8)),
            "GoalDiff": int(m.group(9)), "Points": int(m.group(10)), "Pool": pool_letter,
        })
    return rows


def scrape_world_cup_gender(gender_label):
    """Clicks through Pool A-D for whichever gender tab is currently
    selected, reporting clearly which pools were found vs not \u2014
    consistent with the diagnostic-first approach used elsewhere in
    this project, since the exact tab click mechanism could not be
    tested against a live session before this was written."""
    all_rows = []
    seen_team_sets = []  # confirmed real bug guard, see below
    for pool_letter in ["A", "B", "C", "D"]:
        clicked = False
        try:
            robust_click_text(f"Pool {pool_letter}")
            clicked = True
        except Exception:
            clicked = False
        if not clicked:
            print(f"  \u26a0 {gender_label} Pool {pool_letter}: could not click this pool's tab, skipping.")
            # Real diagnostic, not a guess: if this is still failing,
            # this shows exactly what IS on the page for "Pool
            # {pool_letter}", so the next fix can be based on real
            # evidence instead of trying another selector blindly.
            if pool_letter == "A":
                page_source = driver.page_source
                contains_text = f"Pool {pool_letter}" in page_source
                print(f"      DIAGNOSTIC: page source contains the text 'Pool {pool_letter}': {contains_text}")
                matching_elements = driver.find_elements(By.XPATH, f"//*[contains(text(), 'Pool {pool_letter}')]")
                print(f"      DIAGNOSTIC: {len(matching_elements)} elements contain that text")
                for el in matching_elements[:3]:
                    try:
                        print(f"      DIAGNOSTIC: tag={el.tag_name}, visible={el.is_displayed()}, "
                              f"class={el.get_attribute('class')!r}")
                    except Exception:
                        pass
            continue
        time.sleep(2)
        pool_rows = scrape_world_cup_pool(pool_letter)

        # Confirmed real bug, fixed: on a real run, every pool returned
        # the identical four teams and identical results \u2014 the click
        # did not throw an error, but the page content genuinely never
        # changed. A click "succeeding" without an exception is not the
        # same as the click actually working, so this checks the real
        # result: if this pool's teams exactly match an already-seen
        # pool's teams (for this same gender), the content did not
        # change, and keeping it would silently duplicate one real
        # pool\'s data across all four labels \u2014 worse than having
        # less data, since it would look complete while being wrong.
        team_set = frozenset(r["Team"] for r in pool_rows)
        if pool_rows and team_set in seen_team_sets:
            print(f"  \u26a0 {gender_label} Pool {pool_letter}: same teams as an already-captured pool \u2014 "
                  f"the click likely did not change the page. Skipping rather than saving duplicate data.")
            # Real diagnostic: something WAS found and clicked without
            # error, but the page didn\'t actually change \u2014 dumping
            # the real outerHTML of every matching element (not just
            # tag/class) to see exactly what is being clicked, since
            # that\'s the only way to tell a decoy element from the
            # genuine one without guessing again.
            if pool_letter == "B":
                matching_elements = driver.find_elements(By.XPATH, f"//*[contains(text(), 'Pool {pool_letter}')]")
                print(f"      DIAGNOSTIC: {len(matching_elements)} elements contain the text 'Pool {pool_letter}'")
                for j, el in enumerate(matching_elements[:5]):
                    try:
                        outer_html = driver.execute_script("return arguments[0].outerHTML;", el)
                        print(f"      DIAGNOSTIC element {j}: {outer_html[:300]}")
                    except Exception as e:
                        print(f"      DIAGNOSTIC element {j}: could not read ({e})")
            continue
        seen_team_sets.append(team_set)

        print(f"  {gender_label} Pool {pool_letter}: {len(pool_rows)} teams found"
              + (" (0-0 across the board is expected if the tournament hasn't started yet)" if pool_rows and all(r["Played"] == 0 for r in pool_rows) else ""))
        all_rows.extend(pool_rows)
    return all_rows


print("scrape_world_cup_gender() ready.")


scrape_world_cup_gender() ready.


In [5]:
# Cell WC2 - Run World Cup Scraper (Both Genders)

driver.get(WORLD_CUP_URL)
try:
    WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
except TimeoutException:
    raise RuntimeError(f"Page did not load in time: {WORLD_CUP_URL}")
time.sleep(3)

print("Scraping Men's pools (default view)...")
mens_rows = scrape_world_cup_gender("Men's")
for r in mens_rows:
    r["Gender"] = "Men"

print("\nAttempting to switch to Women's view...")
womens_rows = []
women_clicked = False
try:
    robust_click_text("Women's")
    women_clicked = True
except Exception:
    women_clicked = False

# Real diagnostic: the click reports success (no exception), but the
# very next scrape finds zero "Pool A" content anywhere \u2014 a strong
# sign this may be clicking the wrong element entirely, since
# PARTIAL_LINK_TEXT "Women" can match anything containing that word,
# not just the gender toggle (a news headline, a team name, etc.).
# Dumping every real match here shows exactly what was actually
# clicked, rather than assuming it was the right one.
if women_clicked:
    all_women_matches = driver.find_elements(By.XPATH, "//*[contains(text(), 'Women')]")
    print(f"DIAGNOSTIC: {len(all_women_matches)} elements on the page contain the text 'Women' "
          f"(the click used whichever one Selenium found first, not necessarily the gender toggle)")
    for j, el in enumerate(all_women_matches[:5]):
        try:
            outer_html = driver.execute_script("return arguments[0].outerHTML;", el)
            print(f"DIAGNOSTIC element {j}: {outer_html[:250]}")
        except Exception:
            pass

if women_clicked:
    time.sleep(2)
    print("Scraping Women's pools...")
    womens_rows = scrape_world_cup_gender("Women's")
    for r in womens_rows:
        r["Gender"] = "Women"
else:
    print("Could not find/click a Women's tab \u2014 only Men's data captured this run.")

world_cup_df = pd.DataFrame(mens_rows + womens_rows)
print(f"\nTotal rows captured: {len(world_cup_df)}")
world_cup_df


Scraping Men's pools (default view)...
  ⚠ Men's Pool A: could not click this pool's tab, skipping.


      DIAGNOSTIC: page source contains the text 'Pool A': True
      DIAGNOSTIC: 9 elements contain that text
      DIAGNOSTIC: tag=script, visible=False, class=''
      DIAGNOSTIC: tag=p, visible=False, class='team-time-text'
      DIAGNOSTIC: tag=p, visible=False, class='pool venue'
  ⚠ Men's Pool B: could not click this pool's tab, skipping.


  ⚠ Men's Pool C: could not click this pool's tab, skipping.
  ⚠ Men's Pool D: could not click this pool's tab, skipping.

Attempting to switch to Women's view...


DIAGNOSTIC: 10 elements on the page contain the text 'Women' (the click used whichever one Selenium found first, not necessarily the gender toggle)
DIAGNOSTIC element 0: <script data-n-head="ssr">
            window.undefinedSelectorList = window.undefinedSelectorList || [];
            window.undefinedSelectorList.push("undefined")
            window.configData = "%7B%22status%22%3A200%2C%22ApplicationDomain%22%3Anu
DIAGNOSTIC element 1: <h4 class="event-highlight-name">FIH Women's Nations Cup 2025-26</h4>
DIAGNOSTIC element 2: <span class="nav-text">Women in Hockey</span>
DIAGNOSTIC element 3: <span>Women</span>
DIAGNOSTIC element 4: <h2 class="article-title">
                  FIH Hockey Women's World Cup 2026: JSW Save of the Day - Jessica Buchanan (SCO) vs ARG | #HWC2026</h2>


Scraping Women's pools...


  Women's Pool A: 4 teams found


  Women's Pool B: 4 teams found


  Women's Pool C: 4 teams found


  Women's Pool D: 4 teams found

Total rows captured: 16


,PoolRank,Team,Played,Won,Drawn,Lost,ScoredFor,ScoredAgainst,GoalDiff,Points,Pool,Gender
0,1,Netherlands,3,3,0,0,15,2,13,9,A,Women
1,2,Australia,3,1,1,1,8,8,0,4,A,Women
2,3,Japan,3,1,0,2,3,11,-8,3,A,Women
3,4,Chile,3,0,1,2,4,9,-5,1,A,Women
4,1,Argentina,3,2,1,0,6,3,3,7,B,Women
5,2,Germany,3,2,0,1,8,4,4,6,B,Women
6,3,United States,3,1,1,1,5,6,-1,4,B,Women
7,4,Scotland,3,0,0,3,2,8,-6,0,B,Women
8,1,Belgium,3,2,1,0,8,4,4,7,C,Women
9,2,Spain,3,2,1,0,7,3,4,7,C,Women


In [6]:
# Cell WC3 - Save World Cup Standings

world_cup_df.to_csv("../data/processed/world_cup_standings.csv", index=False)
print("Saved world_cup_standings.csv")


Saved world_cup_standings.csv


## Module: World Cup Team Goal-Type Stats \u2014 The Kenya Comparison

Confirmed directly on FIH's own Stats page: a "Team Statistics" tab
tracks Goals, Field Goals (FG), Penalty Corners (PC), Penalty Strokes
(PS), and Penalty Strokes Missed (PSM) per team \u2014 the same
breakdown already computed for Kenya's own international matches in
`04_international.ipynb`. This makes a direct, honest comparison
possible: not Kenya against a World Cup team's overall results, but
Kenya's actual scoring pattern against genuine World Cup-level teams,
specifically for national-team selectors to see where Kenya's game
differs tactically, not just in outcome.

**Honest note: the click path to this specific tab (Stats \u2192 Team
Statistics) was not tested against a live session**, the same real
limitation as the rankings and pool-standings scrapers elsewhere in
this project. Diagnostic output below reports clearly what worked.


In [7]:
# Cell WC4 - World Cup Team Goal-Type Stats Scraper

WORLD_CUP_STATS_URL = "https://www.fih.hockey/events/fih-hockey-worldcup-belgium-netherlands-2026/stats"

def scrape_world_cup_team_stats(gender_label):
    """Clicks to the Team Statistics tab for whichever gender is
    currently selected and extracts each team's FG/PC/PS breakdown."""
    clicked = False
    try:
        robust_click_text("Team Statistics")
        clicked = True
    except Exception:
        clicked = False

    if not clicked:
        print(f"  \u26a0 {gender_label}: could not click 'Team Statistics' tab.")
        return []

    time.sleep(2)
    page_text = BeautifulSoup(driver.page_source, "html.parser").get_text(" ", strip=True)

    # Confirmed real column order from FIH's own glossary: Name, Goals,
    # FG, PC, PS, PSM
    row_pattern = re.compile(
        r"([A-Z][A-Za-z .]+?)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)"
    )
    # Confirmed real bug, fixed: the team-name group is greedy enough
    # to swallow adjacent column-header text before reaching the first
    # real team, producing a fake "team" like "Penalty Strokes Scored
    # Teams GS GC FGS PCS PSS..." on a real run. A genuine team name is
    # never more than a few words and never contains the header labels
    # themselves, so both are used here as a real, checked filter
    # rather than trusting every regex match blindly.
    HEADER_WORDS = {"penalty", "strokes", "scored", "teams", "team", "gs", "gc", "fgs", "pcs", "pss", "goals"}
    rows = []
    for m in row_pattern.finditer(page_text):
        team_name = m.group(1).strip()
        words = team_name.split()
        looks_like_header = len(words) > 4 or any(w.lower() in HEADER_WORDS for w in words)
        if looks_like_header:
            continue
        goals, field_goals, corners, strokes = int(m.group(2)), int(m.group(3)), int(m.group(4)), int(m.group(5))
        # Confirmed real bug, fixed: a real, downstream comparison card
        # (Kenya's Scoring Pattern vs. World Cup Level) was showing an
        # impossible 213.3% share for a goal type, traced back to here
        # \u2014 any single goal-type count can never exceed the team's
        # total goals, so a row where it does is a parsing misalignment,
        # not real data, and should never have been kept in the first
        # place.
        if field_goals > goals or corners > goals or strokes > goals:
            continue
        rows.append({
            "Team": team_name, "Goals": goals,
            "FieldGoals": field_goals, "PenaltyCorners": corners,
            "PenaltyStrokes": strokes, "PenaltyStrokesMissed": int(m.group(6)),
            "Gender": gender_label,
        })
    print(f"  {gender_label}: {len(rows)} teams found in Team Statistics")
    return rows


print("scrape_world_cup_team_stats() ready.")


scrape_world_cup_team_stats() ready.


In [8]:
# Cell WC5 - Run Team Stats Scraper (Both Genders)

driver.get(WORLD_CUP_STATS_URL)
try:
    WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
except TimeoutException:
    raise RuntimeError(f"Page did not load in time: {WORLD_CUP_STATS_URL}")
time.sleep(3)

print("Scraping Men's Team Statistics (default view)...")
mens_team_stats = scrape_world_cup_team_stats("Men")

print("Attempting to switch to Women's view...")
womens_team_stats = []
women_clicked = False
try:
    robust_click_text("Women's")
    women_clicked = True
except Exception:
    women_clicked = False

if women_clicked:
    time.sleep(2)
    print("Scraping Women's Team Statistics...")
    womens_team_stats = scrape_world_cup_team_stats("Women")
    # Confirmed real bug, fixed: on a real run, this produced the
    # identical England/India numbers under both Men and Women \u2014
    # the click did not throw an error, but the page genuinely never
    # changed. Same guard as the pool standings scraper: if the
    # "Women's" scrape returns the exact same teams as the Men's
    # scrape just did, the click did not work, and this is silently
    # duplicated Men's data, not real Women's data.
    mens_teams = frozenset(r["Team"] for r in mens_team_stats)
    womens_teams = frozenset(r["Team"] for r in womens_team_stats)
    if womens_team_stats and womens_teams == mens_teams:
        print("  \u26a0 Women's Team Statistics returned the exact same teams as Men's \u2014 "
              "the click likely did not change the page. Discarding this as duplicate Men's data, "
              "not real Women's data.")
        womens_team_stats = []
else:
    print("Could not find/click a Women's tab \u2014 only Men's team stats captured this run.")

world_cup_team_stats_df = pd.DataFrame(mens_team_stats + womens_team_stats)
if len(world_cup_team_stats_df) > 0:
    world_cup_team_stats_df["FGShare"] = (world_cup_team_stats_df["FieldGoals"] / world_cup_team_stats_df["Goals"].replace(0, pd.NA) * 100).round(1)
    world_cup_team_stats_df["PCShare"] = (world_cup_team_stats_df["PenaltyCorners"] / world_cup_team_stats_df["Goals"].replace(0, pd.NA) * 100).round(1)
    world_cup_team_stats_df["PSShare"] = (world_cup_team_stats_df["PenaltyStrokes"] / world_cup_team_stats_df["Goals"].replace(0, pd.NA) * 100).round(1)

print(f"\nTotal team-stat rows captured: {len(world_cup_team_stats_df)}")
world_cup_team_stats_df


Scraping Men's Team Statistics (default view)...


  Men: 7 teams found in Team Statistics
Attempting to switch to Women's view...


Scraping Women's Team Statistics...


  Women: 8 teams found in Team Statistics

Total team-stat rows captured: 15


,Team,Goals,FieldGoals,PenaltyCorners,PenaltyStrokes,PenaltyStrokesMissed,Gender,FGShare,PCShare,PSShare
0,Netherlands,12,3,6,5,1,Men,25.0,50.0,41.7
1,Argentina,11,4,7,3,1,Men,36.4,63.6,27.3
2,India,10,8,3,6,1,Men,80.0,30.0,60.0
3,Belgium,8,6,4,4,0,Men,75.0,50.0,50.0
4,Germany,7,2,5,2,0,Men,28.6,71.4,28.6
5,Spain,6,3,5,1,0,Men,50.0,83.3,16.7
6,Australia,6,3,1,4,1,Men,50.0,16.7,66.7
7,China,11,5,6,4,1,Women,45.5,54.5,36.4
8,India,8,3,5,3,0,Women,37.5,62.5,37.5
9,Belgium,8,4,4,4,0,Women,50.0,50.0,50.0


In [9]:
# Cell WC6 - Save World Cup Team Goal-Type Stats

world_cup_team_stats_df.to_csv("../data/processed/world_cup_team_stats.csv", index=False)
print("Saved world_cup_team_stats.csv")


Saved world_cup_team_stats.csv
